In [ ]:
from agents import Agent, WebSearchTool, trace, Runner, gen_trace_id, function_tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
import os
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from typing import Dict
from IPython.display import display, Markdown



In [ ]:
load_dotenv(override=True)

In [ ]:
os.environ["OPENAI_API_KEY"] = os.environ.get("OPENROUTER_API_KEY")
os.environ["OPENAI_BASE_URL"] = os.environ.get("OPENROUTER_BASE_URL")


In [ ]:

@function_tool
def send_email(to_email: str, subject: str, body: str) -> Dict[str, str]:
    """
    Send an email using Gmail SMTP.
    """
    to_email = "sirphil.bizz@gmail.com"
    try:
        msg = MIMEMultipart("alternative")
        msg["From"] = os.environ.get("EMAIL")
        msg["To"] = to_email
        msg["Subject"] = subject

        # Support both plain text and HTML
        msg.attach(MIMEText(body, "plain"))
        msg.attach(MIMEText(body, "html"))

        print(f"Connecting to Gmail SMTP to send to {to_email}...")
        server = smtplib.SMTP("smtp.gmail.com", 587)
        server.starttls()

        email_env = os.environ.get("EMAIL")
        app_password_env = os.environ.get("APP_PASSWORD")

        if not email_env or not app_password_env:
            return {"status": "error", "message": "EMAIL or APP_PASSWORD env variable not set"}

        server.login(email_env, app_password_env)
        server.send_message(msg)
        server.quit()

        print("Email sent successfully!")
        return {"status": "success", "message": f"Email sent to {to_email}"}

    except Exception as e:
        print("Failed to send email:", e)
        return {"status": "error", "message": str(e)}

In [ ]:
class InvestorEmail(BaseModel):
    subject: str = Field(description="Email subject line")
    email_body: str = Field(description="Personalized investor outreach email")

In [ ]:

@function_tool
def web_search(query: str) -> str:
    """
    Custom web search tool for the AI agent.
    Currently returns a placeholder summary.
    Replace with any real web search API later.
    """
    return f"Top results for '{query}':\n- Investment focus: AI and fintech\n- Portfolio companies: Startup A, Startup B\n- Industries: SaaS, Fintech\n- Likely interest: Early-stage AI startups"

In [ ]:

research_agent = Agent(
    name="Investor Research Agent",
    instructions="""
You are an expert venture capital analyst.

Research the investor using web search and summarize:
- Investment focus
- Portfolio companies
- Industries they invest in
- Why they might invest in the startup
""",
    tools=[web_search],  
    model="openai/gpt-4o-mini",
)

In [ ]:
email_agent = Agent(
    name="Investor Email Agent",
    instructions="""
You are a startup founder writing to an investor.

Use the research provided to:
- Personalize the message
- Explain why the investor is a good fit
- Pitch the startup
- Keep it concise and compelling
""",
    model="openai/gpt-4o-mini",
    output_type=InvestorEmail
)

In [ ]:
outreach_agent = Agent(
    name="Outreach Agent",
    instructions="""
You send investor outreach emails.
Use the send_email tool to send the message.
""",
    tools=[send_email],
    model="openai/gpt-4o-mini",
    model_settings=ModelSettings(tool_choice="required"),
)

In [ ]:
async def run_agent():

    investor = input("Investor name: ")
    startup = input("Describe your startup: ")

    print("\nResearching investor...\n")

    research_result = await Runner.run(
        research_agent,
        f"Research investor {investor} for this startup:\n{startup}"
    )

    display(Markdown("### Investor Research"))
    display(Markdown(research_result.final_output))

    print("\nGenerating email...\n")

    email_result = await Runner.run(
        email_agent,
        f"""
Investor: {investor}

Startup:
{startup}

Research:
{research_result.final_output}
"""
    )

    email = email_result.final_output

    display(Markdown("### Generated Email"))
    display(Markdown(email.email_body))

    print("\nSending email...\n")

    await Runner.run(
        outreach_agent,
        f"""
Send this email.

Subject: {email.subject}

Body:
{email.email_body}
"""
    )


trace_id = gen_trace_id()

with trace("Investor Outreach Agent", trace_id=trace_id):
    # Notebook-friendly await
    await run_agent()